In [5]:
import torch
import numpy as np

from dinosaw.utils import add_custom_font, get_features
from dinosaw.wrappers import get_model, WRAPPER_CHECKPOINTS
from dinosaw.linear_probe import do_linear_probe, RampTypes, LinearProbeResult, get_ramp, gen_sample_mask

from os import listdir
from PIL import Image

import matplotlib.pyplot as plt
from matplotlib.gridspec import GridSpec
from typing import Literal
from matplotlib.lines import Line2D

SEED = 100001
torch.manual_seed(SEED)
np.random.seed(SEED)
DEVICE = 'cuda:0'

In [ ]:
GridType = Literal["grid", "grid_holdout", "random"]
GRID_TYPE:list[GridType] = ["grid", "grid_holdout", "random"]

In [7]:
selected_model = 'alibi_coco_dinov2_s' # 'dv2' 
chkpoint = WRAPPER_CHECKPOINTS.get(selected_model, None)
model = get_model(selected_model, DEVICE, True, f"../../models/checkpoints/{chkpoint}", "../../models/dinov3")

n_dims = model.embed_dim

2026-07-24 10:55:49 | I | factory.py                 : 152 | Building wrapper 'alibi_coco_dinov2_s' on device cuda:0
2026-07-24 10:55:49 | I | factory.py                 : 131 | Building backbone with config: BackboneConfig(backbone_type='timm', model_arch='dinov2_s', pretrained=False, checkpoint_path='../../models/checkpoints/trained/alibi_coco_dv2_vits14_reg_ms.pth', model_conf_path='../../models/dinov3', stride=None, remove_pos_embed=True, add_flash_attn=False, dynamic_img_size=True, dynamic_img_pad=False, modifications=[functools.partial(<function add_alibi at 0x7f3e0597e0c0>, slope_type='constant', n_reg_tokens=4, metric='euclidean', normalize=True, wrap=True, add_cls=True, jitter_mag=0.0)], dtype=torch.float32)
2026-07-24 10:55:50 | I | modifications.py           :  47 | Removed default pos. embed
2026-07-24 10:55:50 | I | factory.py                 :  80 | Loading checkpoint: ../../models/checkpoints/trained/alibi_coco_dv2_vits14_reg_ms.pth
2026-07-24 10:55:50 | I | wrapper.py  

In [8]:
ds_folder = 'data/linear_probe/homog_micros'
image_files = [f for f in listdir(ds_folder)]
n_imgs = len(image_files)

features = []
for img_file in image_files:
    img_path = f'{ds_folder}/{img_file}'
    img = Image.open(img_path).convert('RGB')
    feats = get_features(model, img, channel_last=True)
    features.append(feats)

2026-07-24 10:55:50 | I | wrapper.py                 :  92 | Processing image, size: [699, 578]
2026-07-24 10:55:50 | I | wrapper.py                 : 192 | Forward Features: x: [1,3,574,686] -> f: [1,384,41,49]
2026-07-24 10:55:50 | I | wrapper.py                 :  92 | Processing image, size: [597, 596]
2026-07-24 10:55:50 | I | wrapper.py                 : 192 | Forward Features: x: [1,3,588,588] -> f: [1,384,42,42]
2026-07-24 10:55:50 | I | wrapper.py                 :  92 | Processing image, size: [800, 528]
2026-07-24 10:55:50 | I | wrapper.py                 : 192 | Forward Features: x: [1,3,518,798] -> f: [1,384,37,57]
2026-07-24 10:55:50 | I | wrapper.py                 :  92 | Processing image, size: [631, 512]
2026-07-24 10:55:50 | I | wrapper.py                 : 192 | Forward Features: x: [1,3,504,630] -> f: [1,384,36,45]
2026-07-24 10:55:50 | I | wrapper.py                 :  92 | Processing image, size: [700, 700]
2026-07-24 10:55:50 | I | wrapper.py                 : 1

In [ ]:
ramps: tuple[RampTypes, ...] = ('lr', 'ud', 'radial', 'diag', 'random',)
ramps_to_results: dict[RampTypes, list[LinearProbeResult]] = {grid_type: {r: [] for r in ramps} for grid_type in GRID_TYPE}

for grid_ in GRID_TYPE:
    match grid_:
        case "random":
            MASK_CUTOFF_FRAC = 1
            STEP = 6
            RANDOM_MASK = True
        case "grid":
            MASK_CUTOFF_FRAC = 1
            STEP = 6
            RANDOM_MASK = False
        case "grid_holdout":
            MASK_CUTOFF_FRAC = 0.7
            STEP = 6
            RANDOM_MASK = False
    for ramp in ramps:
        for i in range(n_imgs):
            feats = features[i]
            result = do_linear_probe(feats, ramp, probe_by_channel=True, mask_step=STEP, mask_cutoff_frac=MASK_CUTOFF_FRAC, random_mask=RANDOM_MASK)
            ramps_to_results[grid_][ramp].append(result)

In [10]:
from skimage.transform import resize
def average_results(results: list[LinearProbeResult]) -> tuple[np.ndarray, np.ndarray, float, float, np.ndarray]:
    channel_scores_arr = np.array([res['per_channel_scores'] for res in results])
    mean_channel_scores = np.mean(channel_scores_arr, axis=0)
    std_channel_scores = np.std(channel_scores_arr, axis=0)

    scores_arr = np.array([res['stack_r_squared'] for res in results])
    mean_score = np.mean(scores_arr, axis=0)
    std_score = np.std(scores_arr, axis=0)

    n_pred_dims = results[0]['stack_pred'].shape[-1]
    mean_pred = np.zeros((34, 34, n_pred_dims))

    for res in results:
        pred = resize(res['stack_pred'], mean_pred.shape, order=1)
        mean_pred += pred / len(results)

    return mean_channel_scores, std_channel_scores, mean_score, std_score, mean_pred

In [40]:
from matplotlib.patches import Rectangle
from matplotlib.collections import PatchCollection

def add_red_square_overlay(ax: plt.Axes, mask: np.ndarray, sf_h: float, sf_w: int) -> None:
    patches: list[Rectangle] = []
    y_inds, x_inds  = np.nonzero(mask)
    for x, y in zip(x_inds, y_inds):
        rect = Rectangle((x - sf_w / 2, y - sf_h / 2), sf_w, sf_h)
        patches.append(rect)
    pc = PatchCollection(patches, color='red', facecolor='none', lw=0.25)
    ax.add_collection(pc)

In [41]:

n_rows, n_cols = len(ramps), 8


add_custom_font('resources/fonts', 'Grotesk')
W, H =  7.5, 2.5 * 2.3

w_spacing = [2 for _ in range(n_cols)]
SPACE_ROW_IDXS = (4,)

FIG_B_COL_OFFSET = 1
FIG_B_W_COLS = 0
FIG_C_COL_OFFSET = FIG_B_COL_OFFSET + FIG_B_W_COLS + 2

fig = plt.figure(figsize=(W, H))
gs = GridSpec(n_rows, n_cols, figure=fig, width_ratios=w_spacing, wspace=0.8)
colors: dict[RampTypes, str] = {
    'lr': '#5762D5',
    'ud': '#6370C0',
    'diag': '#6E7DAB',
    'radial': '#575366',
    'random': "#2F2D38",
}
ramp_to_title: dict[RampTypes, str] = {
    'lr': 'Left-right',
    'ud': 'Up-down',
    'diag': 'Diagonal',
    'radial': 'Radial',
    'random': 'Random',
}


top_left_ramp_ax = None

for col, grid_ in enumerate(GRID_TYPE):
    match grid_:
        case "random":
            MASK_CUTOFF_FRAC = 1
            STEP = 6
            RANDOM_MASK = True
        case "grid":
            MASK_CUTOFF_FRAC = 1
            STEP = 6
            RANDOM_MASK = False
        case "grid_holdout":
            MASK_CUTOFF_FRAC = 0.7
            STEP = 6
            RANDOM_MASK = False
    for row, ramp in enumerate(ramps):
        h, w = 34, 34
        ramp_arr = get_ramp(ramp, h, w)
        ramp_ax = fig.add_subplot(gs[row, 0 + 2*col])
        ramp_ax.imshow(ramp_arr, cmap='viridis', vmin=0, vmax=1)

        mask = gen_sample_mask((h, w), ramp, STEP, MASK_CUTOFF_FRAC, random_mask=RANDOM_MASK)
        add_red_square_overlay(ramp_ax, mask, 1, 1)

        ramp_ax.set_xticks([])
        ramp_ax.set_yticks([])

        ramp_ax.set_ylabel(ramp_to_title[ramp], labelpad=0, fontsize=8)

        if row == 0:
            title = grid_ if grid_ != "grid_holdout" else "holdout"
            ramp_ax.set_title(f"'{title}'", fontsize=10 )
            top_left_ramp_ax = ramp_ax

        mean_channel_scores, std_channel_scores, mean_score, _, mean_pred = average_results(ramps_to_results[grid_][ramp])

        mean_pred_ax = fig.add_subplot(gs[row, FIG_B_COL_OFFSET + FIG_B_W_COLS + 2*col])
        mean_pred_ax.imshow(mean_pred, cmap='viridis', vmin=0, vmax=1)

        if row == 0:
            mean_pred_ax.set_title('Mean pred.',  fontsize=10 )
        

        mean_pred_ax.set_ylabel(f'$R^{2}:${mean_score:.2f}', labelpad=0, fontsize=8 )
        mean_pred_ax.set_xticks([])
        mean_pred_ax.set_yticks([])

pos2 = fig.axes[1].get_position()  # End of column 2
pos3 = fig.axes[10].get_position()

pos4 = fig.axes[11].get_position()  # End of column 4
pos5 = fig.axes[20].get_position()

# positioning line inbetween
x_sep1 = (pos2.x1 + pos3.x0) / 2 -0.003 
x_sep2 = (pos4.x1 + pos5.x0) / 2 -0.003

for x in [x_sep1, x_sep2]:
    fig.add_artist(Line2D(
        [x, x], [0.1, 0.86],
        transform=fig.transFigure,
        color="black",
        linewidth=1
    ))



SAVE = True
if SAVE:
    plt.savefig('saved/S2.pdf', dpi=300, bbox_inches='tight')
    plt.close()

findfont: Failed to find font weight normal, now using 300.


findfont: Failed to find font weight normal, now using 300.
findfont: Failed to find font weight normal, now using 300.
